# Extração — camada Bronze

Lê o CSV com esquema explícito, preserva cada snapshot recebido e registra lote, arquivo, horário de ingestão e identificador técnico da linha. Informe `source_path` e `batch_id` nos parâmetros do notebook. Reexecutar um lote já carregado não insere os dados novamente.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
dbutils.widgets.text("source_path", "/Workspace/Users/brunoagra988@gmail.com/databricks_analysis_project/data/EmendasParlamentares.csv")
dbutils.widgets.text("batch_id", "emendas_2026-11-08_v1")

source_path = dbutils.widgets.get("source_path").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not source_path:
    raise ValueError("Informe o caminho do CSV no parâmetro source_path.")
if not batch_id:
    raise ValueError("Informe um identificador único da versão do arquivo em batch_id.")

In [ ]:
import re
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructField, StructType, StringType

tabela_bronze = "emendas_parlamentares_bronze"
colunas_origem = (
    "Código da Emenda",
    "Ano da Emenda",
    "Tipo de Emenda",
    "Código do Autor da Emenda",
    "Nome do Autor da Emenda",
    "Número da emenda",
    "Localidade de aplicação do recurso",
    "Código Município IBGE",
    "Município",
    "Código UF IBGE",
    "UF",
    "Região",
    "Código Função",
    "Nome Função",
    "Código Subfunção",
    "Nome Subfunção",
    "Código Programa",
    "Nome Programa",
    "Código Ação",
    "Nome Ação",
    "Código Plano Orçamentário",
    "Nome Plano Orçamentário",
    "Valor Empenhado",
    "Valor Liquidado",
    "Valor Pago",
    "Valor Restos A Pagar Inscritos",
    "Valor Restos A Pagar Cancelados",
    "Valor Restos A Pagar Pagos",
)

def limpar_nome_coluna(nome):
    nome_limpo = re.sub(r"[ ,;{}()\n\t=]", "_", nome)
    return re.sub(r"_+", "_", nome_limpo).strip("_")

colunas_bronze = [limpar_nome_coluna(nome) for nome in colunas_origem]
schema_origem = StructType(
    [StructField(nome, StringType(), True) for nome in colunas_origem]
)

if batch_id.lower() == "legacy":
    raise ValueError("O valor 'legacy' é reservado para migração do esquema Bronze anterior.")

lote_existente = False
bronze_existente = None
if spark.catalog.tableExists(tabela_bronze):
    bronze_existente = spark.table(tabela_bronze)
    if "source_batch_id" in bronze_existente.columns:
        lote_existente = (
            bronze_existente
            .where(F.col("source_batch_id") == batch_id)
            .limit(1)
            .count() > 0
        )

if lote_existente:
    print(f"Lote {batch_id} já existe em {tabela_bronze}; nenhuma linha foi inserida.")
else:
    df_emendas = (
        spark.read.format("csv")
        .schema(schema_origem)
        .option("header", "true")
        .option("enforceSchema", "false")
        .option("mode", "FAILFAST")
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .load(source_path)
        .toDF(*colunas_bronze)
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("source_batch_id", F.lit(batch_id))
        .withColumn("Data_Carga", F.current_timestamp())
    )

    def adicionar_id_linha(df):
        hash_linha = F.sha2(
            F.to_json(F.struct(*[F.col(nome) for nome in colunas_bronze + ["source_file"]])),
            256,
        )
        df = df.withColumn("_hash_linha", hash_linha)
        ordem_duplicatas = Window.partitionBy("_hash_linha").orderBy(F.col("source_file"))
        return (
            df.withColumn("_ocorrencia_igual", F.row_number().over(ordem_duplicatas))
            .withColumn(
                "source_row_id",
                F.sha2(
                    F.concat_ws(
                        "|", F.col("_hash_linha"), F.col("_ocorrencia_igual").cast("string")
                    ),
                    256,
                ),
            )
            .drop("_hash_linha", "_ocorrencia_igual")
        )

    df_emendas = adicionar_id_linha(df_emendas)

    total_lote = df_emendas.count()
    if total_lote == 0:
        raise ValueError(f"O arquivo {source_path} não contém registros para ingestão.")

    modo_escrita = "append"
    df_gravar = df_emendas
    if bronze_existente is not None and "source_batch_id" not in bronze_existente.columns:
        colunas_necessarias = set(colunas_bronze)
        colunas_ausentes = colunas_necessarias - set(bronze_existente.columns)
        if colunas_ausentes:
            raise ValueError(
                "A Bronze existente não corresponde ao esquema esperado. "
                f"Colunas ausentes: {sorted(colunas_ausentes)}"
            )
        colunas_extras = set(bronze_existente.columns) - colunas_necessarias - {"Data_Carga"}
        if colunas_extras:
            raise ValueError(
                "Migração Bronze interrompida para preservar colunas desconhecidas: "
                f"{sorted(colunas_extras)}"
            )

        legado = bronze_existente.select(
            *[F.col(nome).cast("string").alias(nome) for nome in colunas_bronze],
            F.lit(None).cast("string").alias("source_file"),
            F.lit("legacy").alias("source_batch_id"),
            F.col("Data_Carga").cast("timestamp").alias("Data_Carga")
            if "Data_Carga" in bronze_existente.columns
            else F.current_timestamp().alias("Data_Carga"),
        )
        legado = adicionar_id_linha(legado)
        df_gravar = legado.unionByName(df_emendas)
        modo_escrita = "overwrite"

    (
        df_gravar.write.format("delta")
        .mode(modo_escrita)
        .option("mergeSchema", "true")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_bronze)
    )
    print(f"Lote {batch_id} carregado em {tabela_bronze}: {total_lote} registros.")